# ChurnGuard AI - pipeline walkthrough
Run from the `notebooks/` folder. Uses real data if `data/raw/` has the IBM CSV, otherwise the synthetic test file (results then are **not** real-world).

In [ ]:
import os, sys
os.chdir('..'); sys.path.insert(0, os.getcwd())
import pandas as pd
from src import data_loader, predict
from src.preprocessing import clean_data, outlier_report, class_balance
raw, source = data_loader.load_raw_data()
print(source, raw.shape)

In [ ]:
df, report = clean_data(raw)
print(report)
print(class_balance(df['Churn']))
print(outlier_report(df))

In [ ]:
print(df.groupby('Contract')['Churn'].mean().round(3))
print(df.groupby(pd.cut(df['tenure'], [-1,12,24,48,72,200]), observed=True)['Churn'].mean().round(3))

## Training
`python -m src.train` (about 2 minutes). Then load the saved artifacts:

In [ ]:
meta = predict.load_metadata()
print(meta['best_model'], meta['data_source'])
pd.DataFrame(meta['models'])[['model','cv_recall','cv_f1','cv_roc_auc','recall','f1','roc_auc']].round(3)

In [ ]:
model = predict.load_model()
scored = predict.predict_dataframe(df.drop(columns=['Churn']).head(10), model)
scored[['customerID','ChurnProbability','RiskLevel']]